# Alinhamento OBD-II + LLM com Cross-Attention (Flamingo style)

Notebook para reproduzir a metodologia de alinhamento multimodal no estilo dos autores: encoder temporal + camadas de cross-attention no LLM (OpenTSLMFlamingo).

## 1) Dependências
Execute se necessário.

In [13]:
# %pip install -q -r requirements.txt
# %pip install -q -e .

## 2) Configuração

In [14]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (CWD, *CWD.parents) if (path / 'pyproject.toml').exists()), CWD)
for extra_path in (PROJECT_ROOT, PROJECT_ROOT / 'src', PROJECT_ROOT / 'open_flamingo', PROJECT_ROOT / 'src' / 'open_flamingo'):
    if extra_path.exists() and str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from transformers import get_linear_schedule_with_warmup

from opentslm.model.llm.OpenTSLMFlamingo import OpenTSLMFlamingo
from opentslm.system_metrics import SystemMetricsMonitor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

DATA_PATH = PROJECT_ROOT / 'data/obd_cot_gpt5.jsonl'
# LLM_ID = 'google/gemma-3-270m'
LLM_ID = "meta-llama/Llama-3.2-1B"
BATCH_SIZE = 2
NUM_EPOCHS = 200
LR = 2e-4
WARMUP_FRAC = 0.1
GRAD_CLIP = 1.0
CROSS_ATTN_EVERY_N = 1
USE_DEVICE_MAP = False
# MPS: keep fp32 to avoid NaNs
LLM_DTYPE = torch.bfloat16
METRICS_DIR = PROJECT_ROOT / 'results' / 'notebook_metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RUN_SLUG = f"obd_flamingo_{LLM_ID.split('/')[-1].replace('-', '_').replace('.', '_').lower()}"


## 3) Carregar dataset gerado com GPT-5

In [15]:
rows = [json.loads(x) for x in DATA_PATH.read_text(encoding='utf-8').splitlines() if x.strip()]
print(f'Amostras: {len(rows)}')
print('Modelo de geração textual no dataset:', rows[0].get('llm_model', 'n/a'))
print('Status geração:', {r.get('generation_status', 'n/a') for r in rows})
rows[0]

# Filter out 'congested' class
rows = [r for r in rows if r.get('label') != 'congested']
print('Amostras (sem congested):', len(rows))

# Rebuild pre_prompt to remove label leakage (keep two options, but no 'correct' hint)
_NO_LEAK_PROMPT = """You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[{label}]
[{dis}]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST end your response with \"Answer:\"."""

def _build_no_leak_prompt(r):
    label = r.get('label') or ''
    dis = r.get('dissimilar_label') or ''
    return _NO_LEAK_PROMPT.format(label=label, dis=dis)

for r in rows:
    r['pre_prompt'] = _build_no_leak_prompt(r)

# sanity check
print('pre_prompt sample (no leak):')
print(rows[0]['pre_prompt'])


Amostras: 195
Modelo de geração textual no dataset: n/a
Status geração: {'ok'}
Amostras (sem congested): 193
pre_prompt sample (no leak):
You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[economical]
[aggressive]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final s

## 4) Split train/val/test

In [16]:
idx = list(range(len(rows)))
random.shuffle(idx)

n = len(idx)
train_end = max(1, int(0.6*n))
val_end = max(train_end+1, int(0.8*n)) if n > 2 else train_end

train_rows = [rows[i] for i in idx[:train_end]]
val_rows = [rows[i] for i in idx[train_end:val_end]]
test_rows = [rows[i] for i in idx[val_end:]]

if len(val_rows) == 0 and len(train_rows) > 1:
    val_rows = [train_rows.pop()]
if len(test_rows) == 0 and len(train_rows) > 1:
    test_rows = [train_rows.pop()]

print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))

train/val/test: 115 39 39


## 5) Dataset PyTorch no formato esperado pelo OpenTSLMFlamingo

In [17]:
FEATURE_LABELS = [
    'Speed',
    'RPM',
    'Engine Load',
    'Throttle Position',
]

def summarize_series(label, values, n_points=8):
    values = values.detach().cpu().float()
    n = values.numel()
    idx = torch.linspace(0, max(n - 1, 0), steps=min(n_points, n)).long()
    samples = ', '.join(f'{values[i].item():.1f}' for i in idx)
    delta = values[-1].item() - values[0].item()
    if delta > 1e-3:
        trend = 'overall increasing'
    elif delta < -1e-3:
        trend = 'overall decreasing'
    else:
        trend = 'roughly flat'
    return (
        f'{label} series with mean {values.mean().item():.2f}, '
        f'std {values.std(unbiased=False).item():.2f}, '
        f'min {values.min().item():.2f}, max {values.max().item():.2f}, '
        f'trend {trend}, sampled points [{samples}]:'
    )

class OBDAlignmentDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        ts = torch.tensor(r['time_series'], dtype=torch.float32)
        time_series_text = [
            summarize_series(label, series)
            for label, series in zip(FEATURE_LABELS, ts)
        ]
        return {
            'pre_prompt': r['pre_prompt'],
            'time_series_text': time_series_text,
            'post_prompt': r['post_prompt'],
            'answer': r['answer'],
            'time_series': ts,
            'id': r['id'],
        }

train_ds = OBDAlignmentDataset(train_rows)
val_ds = OBDAlignmentDataset(val_rows)
test_ds = OBDAlignmentDataset(test_rows)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)

In [18]:
# Debug (Flamingo) — imprime séries com rótulos
sample = train_ds[0]
print("pre_prompt:", sample["pre_prompt"])
print("time_series_text:", sample["time_series_text"])
print("post_prompt:", sample["post_prompt"])
print("answer:", sample["answer"])
print("time_series shape:", sample["time_series"].shape)

labels = ["Speed", "RPM", "Engine Load", "Throttle Position"]
for i, lbl in enumerate(labels):
    print(f"time series of {lbl}:", sample["time_series"][i, :10].tolist())


pre_prompt: You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[economical]
[aggressive]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST end your response with "Answer:".
time_series_text: ['Speed s

## 6) Inicializar modelo Flamingo temporal
A metodologia segue o padrão dos autores: congelar backbone e treinar encoder temporal + perceiver + gated cross-attention.

In [19]:
model = OpenTSLMFlamingo(
    device=DEVICE,
    llm_id=LLM_ID,
    cross_attn_every_n_layers=CROSS_ATTN_EVERY_N,
    llm_torch_dtype=LLM_DTYPE,
    llm_low_cpu_mem_usage=True,
    use_device_map=USE_DEVICE_MAP,
).to(DEVICE)

trainable = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
print('Parametros treinaveis:', sum(p.numel() for _,p in trainable))

def grouped_params(named_params):
    wd, no_wd = [], []
    for n, p in named_params:
        if 'gated_cross_attn' in n:
            wd.append(p)
        else:
            no_wd.append(p)
    return [
        {'params': wd, 'weight_decay': 0.1},
        {'params': no_wd, 'weight_decay': 0.0},
    ]

optimizer = torch.optim.AdamW(grouped_params(trainable), lr=LR)
total_steps = max(1, NUM_EPOCHS * len(train_loader))
warmup_steps = int(WARMUP_FRAC * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


Flamingo Using device: cuda
Parametros treinaveis: 837832224


In [20]:
# Sanity check: ensure labels are present and finite
batch = next(iter(train_loader))
input_ids, images, attention_mask, labels = model.pad_and_apply_batch(batch, include_labels=False)
print('labels_tokens:', int((labels != -100).sum()))
print('images_finite:', bool(torch.isfinite(images).all()))


labels_tokens: 224
images_finite: True


## 7) Treino + validação (early stopping simples)

In [21]:
best_val = float('inf')
best_path = PROJECT_ROOT / 'data/obd_flamingo_best.pt'
patience = 3
no_improve = 0
history = []
train_metrics_path = METRICS_DIR / f'{RUN_SLUG}_train_samples.csv'
train_summary_path = METRICS_DIR / f'{RUN_SLUG}_train_summary.json'

train_monitor = SystemMetricsMonitor(
    label=f'{RUN_SLUG}_train',
    interval_s=0.5,
    disk_path=PROJECT_ROOT,
    metadata={'notebook': 'obd_flamingo_alignment_training', 'stage': 'train', 'llm_id': LLM_ID, 'device': DEVICE},
).start()
train_monitor.mark('setup', train_batches=len(train_loader), val_batches=len(val_loader), num_epochs=NUM_EPOCHS)

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    train_loss = 0.0
    batch_count = 0
    for step, batch in enumerate(train_loader, start=1):
        step_start = __import__('time').perf_counter()
        optimizer.zero_grad()
        loss = model.compute_loss(batch)
        if not torch.isfinite(loss):
            print('Loss nao-finita detectada. Abortando epoca.')
            train_monitor.mark('train_non_finite', epoch=epoch, step=step)
            break
        loss.backward()
        grad_norm = float(clip_grad_norm_(model.parameters(), GRAD_CLIP))
        optimizer.step()
        scheduler.step()
        loss_value = float(loss.item())
        train_loss += loss_value
        batch_count += 1
        train_monitor.mark(
            'train_step',
            epoch=epoch,
            step=step,
            loss=loss_value,
            grad_norm=grad_norm,
            lr=float(optimizer.param_groups[0]['lr']),
            step_duration_s=__import__('time').perf_counter() - step_start,
        )
    train_loss /= max(1, batch_count)

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for val_step, batch in enumerate(val_loader, start=1):
            vloss = model.compute_loss(batch)
            if not torch.isfinite(vloss):
                print('Val loss nao-finita detectada.')
                train_monitor.mark('val_non_finite', epoch=epoch, step=val_step)
                break
            val_loss += float(vloss.item())
            val_batches += 1
    val_loss /= max(1, val_batches)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    train_monitor.mark('epoch_end', epoch=epoch, train_loss=train_loss, val_loss=val_loss)
    print(f'Epoch {epoch}: train={train_loss:.6f} val={val_loss:.6f}')

    if val_loss < best_val:
        best_val = val_loss
        no_improve = 0
        model.store_to_file(str(best_path))
        train_monitor.mark('checkpoint_saved', epoch=epoch, best_val=best_val, checkpoint_path=str(best_path))
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping acionado.')
            train_monitor.mark('early_stop', epoch=epoch, best_val=best_val)
            break

train_monitor.stop(final_phase='finished', best_val=best_val, checkpoint_path=str(best_path))
train_monitor.to_csv(train_metrics_path)
train_summary_path.write_text(json.dumps(train_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
print('Training metrics saved to', train_metrics_path)
print('Training summary saved to', train_summary_path)


Epoch 1: train=3.521657 val=3.519703
Model saved to C:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_flamingo_best.pt
Epoch 2: train=3.184027 val=2.759086
Model saved to C:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_flamingo_best.pt
Epoch 3: train=2.238981 val=2.037610
Model saved to C:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_flamingo_best.pt
Epoch 4: train=1.758212 val=1.910515
Model saved to C:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_flamingo_best.pt
Epoch 5: train=1.500569 val=1.861181
Model saved to C:\Users\Morsinaldo\Desktop\OpenTSLM\data\obd_flamingo_best.pt
Epoch 6: train=1.273959 val=1.885189
Epoch 7: train=1.048962 val=1.971341
Epoch 8: train=0.769701 val=2.178527
Early stopping acionado.


## 8) Carregar melhor checkpoint e inferência no teste

In [22]:
# model.load_from_file('data/obd_flamingo_best.pt')
model.eval()

preds = []
inference_metrics_path = METRICS_DIR / f'{RUN_SLUG}_inference_samples.csv'
inference_summary_path = METRICS_DIR / f'{RUN_SLUG}_inference_summary.json'

inference_monitor = SystemMetricsMonitor(
    label=f'{RUN_SLUG}_inference',
    interval_s=0.5,
    disk_path=PROJECT_ROOT,
    metadata={'notebook': 'obd_flamingo_alignment_training', 'stage': 'inference', 'llm_id': LLM_ID, 'device': DEVICE},
).start()
inference_monitor.mark('setup', test_batches=len(test_loader))

with torch.no_grad():
    for step, batch in enumerate(test_loader, start=1):
        step_start = __import__('time').perf_counter()
        gen = model.generate(batch, max_new_tokens=256)[0]
        latency = __import__('time').perf_counter() - step_start
        preds.append({
            'id': batch[0]['id'],
            'prediction': gen.strip(),
            'target': batch[0]['answer'],
            'prompt': f"{batch[0]['pre_prompt']} | {batch[0]['time_series_text'][0]} | {batch[0]['post_prompt']}",
        })
        inference_monitor.mark(
            'inference_step',
            step=step,
            inference_latency_s=latency,
            prediction_chars=len(gen.strip()),
            sample_id=batch[0]['id'],
        )

inference_monitor.stop(final_phase='finished', n_predictions=len(preds))
inference_monitor.to_csv(inference_metrics_path)
inference_summary_path.write_text(json.dumps(inference_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
print('Inference metrics saved to', inference_metrics_path)
print('Inference summary saved to', inference_summary_path)

preds


[{'id': 'obd_cot_000088',
  'prediction': 'Speed rises smoothly to a moderate cruise and then tapers down to a brief stop before building again, with no sharp oscillations or rapid surges. RPM follows the same pattern with controlled rises and dips rather than high-amplitude fluctuations. Engine load stays mostly in the mid range with occasional, measured increases, and the throttle is predominantly low with a couple of short, controlled pushes instead of repeated hard inputs. The overall variability is moderate and the signals remain coordinated, indicating smooth control and minimal abrupt maneuvers. Answer: normal',
  'target': 'Speed stays mostly low with gradual ramps and plateaus, peaking only in the mid‑20s to around 30 km/h and spending considerable time near idle, indicating mild acceleration and frequent coasting. RPM largely tracks these modest changes, hovering near idle with brief rises below about 2000, never sustaining high revs that would suggest hard accelerations. Eng

## 9) Métrica simples de validação textual (F1 de tokens)

In [23]:
import re
from difflib import SequenceMatcher

def norm_text(s):
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def tok(s):
    return re.findall(r'\w+', s.lower())

def token_f1(pred, gold):
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    inter = 0
    g_used = [False]*len(g)
    for t in p:
        for i,gt in enumerate(g):
            if not g_used[i] and t == gt:
                inter += 1
                g_used[i] = True
                break
    prec = inter/len(p)
    rec = inter/len(g)
    return 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)

def rouge_l_f1(pred, gold):
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    # LCS
    dp = [[0]*(len(g)+1) for _ in range(len(p)+1)]
    for i in range(1, len(p)+1):
        for j in range(1, len(g)+1):
            if p[i-1] == g[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    prec = lcs/len(p)
    rec = lcs/len(g)
    return 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)

def exact_match(pred, gold):
    return 1.0 if norm_text(pred) == norm_text(gold) else 0.0

def seq_sim(pred, gold):
    return SequenceMatcher(None, norm_text(pred), norm_text(gold)).ratio()

scores = {
    'token_f1': [],
    'rouge_l_f1': [],
    'exact_match': [],
    'seq_sim': [],
}
for x in preds:
    scores['token_f1'].append(token_f1(x['prediction'], x['target']))
    scores['rouge_l_f1'].append(rouge_l_f1(x['prediction'], x['target']))
    scores['exact_match'].append(exact_match(x['prediction'], x['target']))
    scores['seq_sim'].append(seq_sim(x['prediction'], x['target']))

print('Token-F1 medio:', round(float(np.mean(scores['token_f1'])) if scores['token_f1'] else 0.0, 4))
print('ROUGE-L F1 medio:', round(float(np.mean(scores['rouge_l_f1'])) if scores['rouge_l_f1'] else 0.0, 4))
print('Exact-Match medio:', round(float(np.mean(scores['exact_match'])) if scores['exact_match'] else 0.0, 4))
print('SeqSim medio:', round(float(np.mean(scores['seq_sim'])) if scores['seq_sim'] else 0.0, 4))

# Classification metrics (paper-style)
LABELS = ['economical', 'normal', 'aggressive', 'congested']

def extract_label(text):
    if text is None:
        return None
    t = str(text).lower()
    # Prefer explicit Answer: tag
    m = re.findall(r'answer\s*:\s*([a-z_\-]+)', t)
    if m:
        cand = m[-1]
        return cand if cand in LABELS else None
    # Fallback: last label occurrence
    positions = [(t.rfind(lbl), lbl) for lbl in LABELS if t.rfind(lbl) != -1]
    if positions:
        positions.sort()
        return positions[-1][1]
    return None

y_true = []
y_pred = []
for x in preds:
    gt = extract_label(x.get('target'))
    pr = extract_label(x.get('prediction'))
    if gt is None or pr is None:
        continue
    y_true.append(gt)
    y_pred.append(pr)

total = len(y_true)
correct = sum(1 for g,p in zip(y_true, y_pred) if g == p)
accuracy = (correct / total) if total else 0.0

# Macro-F1 over fixed label set
macro_f1 = 0.0
for lbl in LABELS:
    tp = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p == lbl)
    fp = sum(1 for g,p in zip(y_true, y_pred) if g != lbl and p == lbl)
    fn = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p != lbl)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
    macro_f1 += f1
macro_f1 = macro_f1 / len(LABELS) if LABELS else 0.0

print('Accuracy:', round(float(accuracy), 4), f'({correct}/{total})')
print('Macro-F1:', round(float(macro_f1), 4))

# Supplementary semantic and structural metrics for paper reporting.
validity_rows = []
for x in preds:
    pred = str(x.get('prediction', ''))
    pred_label = extract_label(pred)
    has_answer_tag = bool(re.search(r'answer\s*:', pred.lower()))
    non_ascii_ratio = sum(ord(ch) > 127 for ch in pred) / max(len(pred), 1)
    validity_rows.append({
        'id': x.get('id'),
        'has_answer_tag': has_answer_tag,
        'pred_label': pred_label,
        'valid_label': pred_label in LABELS,
        'non_ascii_ratio': float(non_ascii_ratio),
        'prediction_length': len(pred),
    })

validity_metrics = {
    'has_answer_tag_rate': float(np.mean([r['has_answer_tag'] for r in validity_rows])) if validity_rows else 0.0,
    'valid_label_rate': float(np.mean([r['valid_label'] for r in validity_rows])) if validity_rows else 0.0,
    'mean_non_ascii_ratio': float(np.mean([r['non_ascii_ratio'] for r in validity_rows])) if validity_rows else 0.0,
    'mean_prediction_length': float(np.mean([r['prediction_length'] for r in validity_rows])) if validity_rows else 0.0,
}

print('Has Answer tag rate:', round(validity_metrics['has_answer_tag_rate'], 4))
print('Valid label rate:', round(validity_metrics['valid_label_rate'], 4))
print('Mean non-ASCII ratio:', round(validity_metrics['mean_non_ascii_ratio'], 4))

detailed_rows = []
for i, x in enumerate(preds):
    row = dict(x)
    row['token_f1'] = float(scores['token_f1'][i])
    row['rouge_l_f1'] = float(scores['rouge_l_f1'][i])
    row['exact_match'] = float(scores['exact_match'][i])
    row['seq_sim'] = float(scores['seq_sim'][i])
    row.update(validity_rows[i])
    detailed_rows.append(row)

advanced_metrics = {}
try:
    from bert_score import score as bertscore_score
    _, _, bert_f1 = bertscore_score(
        [x['prediction'] for x in preds],
        [x['target'] for x in preds],
        lang='en',
        verbose=False,
    )
    bert_f1 = bert_f1.detach().cpu().numpy().tolist()
    advanced_metrics['bertscore_f1_mean'] = float(np.mean(bert_f1)) if bert_f1 else 0.0
    for row, val in zip(detailed_rows, bert_f1):
        row['bertscore_f1'] = float(val)
    print('BERTScore F1 mean:', round(advanced_metrics['bertscore_f1_mean'], 4))
except Exception as exc:
    advanced_metrics['bertscore_f1_mean'] = None
    advanced_metrics['bertscore_error'] = str(exc)
    print('BERTScore unavailable:', exc)

try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
    pred_emb = emb_model.encode([x['prediction'] for x in preds], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    tgt_emb = emb_model.encode([x['target'] for x in preds], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    cosine_scores = (pred_emb * tgt_emb).sum(axis=1).tolist()
    advanced_metrics['embedding_cosine_mean'] = float(np.mean(cosine_scores)) if cosine_scores else 0.0
    for row, val in zip(detailed_rows, cosine_scores):
        row['embedding_cosine'] = float(val)
    print('Embedding cosine mean:', round(advanced_metrics['embedding_cosine_mean'], 4))
except Exception as exc:
    advanced_metrics['embedding_cosine_mean'] = None
    advanced_metrics['embedding_cosine_error'] = str(exc)
    print('Embedding cosine unavailable:', exc)


Token-F1 medio: 0.46
ROUGE-L F1 medio: 0.2391
Exact-Match medio: 0.0
SeqSim medio: 0.0619
Accuracy: 1.0 (39/39)
Macro-F1: 0.5
Has Answer tag rate: 1.0
Valid label rate: 1.0
Mean non-ASCII ratio: 0.0004


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1 mean: 0.8906
Embedding cosine mean: 0.7909


## 10) Exportar resultados

In [24]:
if LLM_ID == "google/gemma-3-270m":
    model_sufix = "gemma3_270m"
elif LLM_ID == "meta-llama/Llama-3.2-1B":
    model_sufix = "llama3_2_1B"

out_pred = Path(f'data/obd_alignment_test_predictions_{model_sufix}.jsonl')
out_detail = Path(f'data/obd_alignment_test_detailed_{model_sufix}.jsonl')
out_summary = Path(f'data/obd_alignment_test_summary_{model_sufix}.json')

with out_pred.open('w', encoding='utf-8') as f:
    for r in preds:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

with out_detail.open('w', encoding='utf-8') as f:
    for row in detailed_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

summary = {
    'n': len(preds),
    'token_f1_mean': float(np.mean(scores['token_f1'])) if scores['token_f1'] else 0.0,
    'rouge_l_f1_mean': float(np.mean(scores['rouge_l_f1'])) if scores['rouge_l_f1'] else 0.0,
    'exact_match_mean': float(np.mean(scores['exact_match'])) if scores['exact_match'] else 0.0,
    'seq_sim_mean': float(np.mean(scores['seq_sim'])) if scores['seq_sim'] else 0.0,
    'accuracy': float(accuracy),
    'macro_f1': float(macro_f1),
    'validity_metrics': validity_metrics,
    'advanced_metrics': advanced_metrics,
}

out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Arquivo salvo em', out_pred)
print('Metricas detalhadas salvas em', out_detail)
print('Resumo salvo em', out_summary)

Arquivo salvo em data\obd_alignment_test_predictions_llama3_2_1B.jsonl
Metricas detalhadas salvas em data\obd_alignment_test_detailed_llama3_2_1B.jsonl
Resumo salvo em data\obd_alignment_test_summary_llama3_2_1B.json
